In [1]:
import os
import subprocess
import pandas as pd
import numpy as np
import json
from pathlib import Path

In [2]:
# Define paths
BASE_DIR = Path("..").resolve()
RAW_DIR = BASE_DIR / "data" / "raw"
INTERIM_DIR = BASE_DIR / "data" / "interim"

# Files of interest
TGZ_PATH = RAW_DIR / "semeval-dataset.tgz"
SEMEVAL_EXTRACT_DIR = RAW_DIR / "semeval2020_data"
INTERIM_SI_FILE = INTERIM_DIR / "semeval_task1_si_merged.csv"
INTERIM_TC_FILE = INTERIM_DIR / "semeval_task2_tc_merged.csv"
INTERIM_AV_FILE = INTERIM_DIR / "averitec_dev_flattened.csv"

In [3]:
# Ensure directories exist
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(INTERIM_DIR, exist_ok=True)

In [4]:
# Check if data is already processed
if INTERIM_SI_FILE.exists() and INTERIM_TC_FILE.exists() and INTERIM_AV_FILE.exists():
    print("Interim files found. Loading directly from data/interim...")
    df_si = pd.read_csv(INTERIM_SI_FILE)
    df_tc = pd.read_csv(INTERIM_TC_FILE)
    averitec = pd.read_csv(INTERIM_AV_FILE)

else:
    print("Data not found in interim. Starting full download and extraction...")

    # 1. Clone AVeriTeC
    averitec_dir = RAW_DIR / "averitec"
    if not averitec_dir.exists():
        print("Cloning AVeriTeC...")
        subprocess.run(
            [
                "git",
                "clone",
                "https://github.com/MichSchli/AVeriTeC.git",
                str(averitec_dir),
            ]
        )

    # 2. Download SemEval
    if not TGZ_PATH.exists():
        print("Downloading SemEval-2020 Task 11 dataset...")
        subprocess.run(
            [
                "curl",
                "-L",
                "-o",
                str(TGZ_PATH),
                "https://zenodo.org/record/3952415/files/datasets-v2.tgz?download=1",
            ]
        )

    # 3. Clone HQP
    ###Not using for now but may later if we can figure out Twitter API
    # hqp_dir = RAW_DIR / "hqp"
    # if not hqp_dir.exists():
    # print("Cloning HQP Dataset Repo...")
    # subprocess.run(["git", "clone", "https://github.com/abdumaa/HiQualProp.git", str(hqp_dir)])

    # Extract SemEval data from TGZ
    if not SEMEVAL_EXTRACT_DIR.exists():
        print("Extracting SemEval tarball...")
        os.makedirs(SEMEVAL_EXTRACT_DIR, exist_ok=True)
        subprocess.run(["tar", "-xvzf", str(TGZ_PATH), "-C", str(SEMEVAL_EXTRACT_DIR)])

    # Connect article texts to labels in SemEval rows
    txt_paths = list(SEMEVAL_EXTRACT_DIR.rglob("*.txt"))

    # Create a dictionary mapping: article_id -> full_text
    article_text_map = {}
    for p in txt_paths:
        article_id = p.stem.replace("article", "")  # Keeps just the number
        with open(p, "r", encoding="utf-8") as f:
            article_text_map[article_id] = f.read()

    # Task 1: Span Identification
    si_paths = list(SEMEVAL_EXTRACT_DIR.rglob("*.task1-SI.labels"))
    df_si = pd.concat(
        [
            pd.read_csv(
                p, sep="\t", header=None, names=["article_id", "start_char", "end_char"]
            ).assign(source_file=p.name)
            for p in si_paths
        ],
        ignore_index=True,
    )

    # Task 2: Technique Classification
    tc_paths = list(SEMEVAL_EXTRACT_DIR.rglob("*.task2-TC.labels"))
    df_tc = pd.concat(
        [
            pd.read_csv(
                p,
                sep="\t",
                header=None,
                names=["article_id", "technique", "start_char", "end_char"],
            ).assign(source_file=p.name)
            for p in tc_paths
        ],
        ignore_index=True,
    )

    # Map the text to the dataframes using the article_id
    df_si["article_id"] = df_si["article_id"].astype(str)
    df_tc["article_id"] = df_tc["article_id"].astype(str)

    df_si["text_content"] = df_si["article_id"].map(article_text_map)
    df_tc["text_content"] = df_tc["article_id"].map(article_text_map)

    # Process AVeriTeC
    with open(RAW_DIR / "averitec/data/dev.json", "r") as f:
        averitec = pd.json_normalize(json.load(f))

    # Save to interim
    df_si.to_csv(INTERIM_SI_FILE, index=False)
    df_tc.to_csv(INTERIM_TC_FILE, index=False)
    averitec.to_csv(INTERIM_AV_FILE, index=False)
    print(f"All files processed and saved to {INTERIM_DIR}")

Interim files found. Loading directly from data/interim...


In [5]:
# Validate it worked by displaying SemEval Span Identification data
df_si.head()

,article_id,start_char,end_char,source_file,text_content
0,999001293,409,427,article999001293.task1-SI.labels,Top Florida County Election Official Illegally...
1,999001293,429,535,article999001293.task1-SI.labels,Top Florida County Election Official Illegally...
2,999001293,380,406,article999001293.task1-SI.labels,Top Florida County Election Official Illegally...
3,999001293,2307,2314,article999001293.task1-SI.labels,Top Florida County Election Official Illegally...
4,999001293,2849,2991,article999001293.task1-SI.labels,Top Florida County Election Official Illegally...


In [6]:
# Display SemEval Technique Classification
df_tc.head()

,article_id,technique,start_char,end_char,source_file,text_content
0,758756657,Repetition,5024,5036,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...
1,758756657,Repetition,5302,5314,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...
2,758756657,Loaded_Language,62,69,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...
3,758756657,Causal_Oversimplification,606,746,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...
4,758756657,Loaded_Language,4352,4361,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...


In [7]:
# Display AveriTec fact-checking claims dataframe
averitec.head()

,claim,required_reannotation,label,justification,claim_date,speaker,original_claim_url,fact_checking_article,reporting_source,location_ISO_code,claim_types,fact_checking_strategies,questions,cached_original_claim_url
0,"In a letter to Steve Jobs, Sean Connery refuse...",False,Refuted,The answer and sources show that the claim was...,31-10-2020,NaN,NaN,https://web.archive.org/web/20201130144023/htt...,Facebook,NaN,['Event/Property Claim'],['Written Evidence'],[{'question': 'Where was the claim first publi...,NaN
1,Trump Administration claimed songwriter Billie...,False,Refuted,Seems that the Wzshington post accused the sin...,31-10-2020,NaN,NaN,https://web.archive.org/web/20201103001419/htt...,Instagram,US,"['Position Statement', 'Event/Property Claim']",['Written Evidence'],[{'question': 'Has the Trump administration vo...,NaN
2,Due to Imran Khan's criticism of Macron's comm...,False,Refuted,The tweet was not the official government page...,31-10-2020,Consulate General Of Pakistan France,https://web.archive.org/web/20201113115127/htt...,https://web.archive.org/web/20210629013122/htt...,Twitter,FR,"['Causal Claim', 'Event/Property Claim']",['Written Evidence'],[{'question': 'How did Macron criticise Islam?...,https://web.archive.org/web/20201113115127/htt...
3,UNESCO declared Nadar community as the most an...,False,Refuted,This claim is refuted. According to the QA pai...,31-10-2020,Kumar Shankar,NaN,https://web.archive.org/web/20210225110220/htt...,Facebook,IN,['Event/Property Claim'],['Written Evidence'],"[{'question': 'What is Nadar?', 'answers': [{'...",NaN
4,Republican Matt Gaetz was part of a company th...,True,Refuted,The company was sold in 2004 and the law suit ...,31-10-2020,NaN,NaN,https://web.archive.org/web/20210713185816/htt...,Facebook,US,"['Numerical Claim', 'Event/Property Claim']","['Written Evidence', 'Numerical Comparison']",[{'question': 'Did Matt Gaetz work for Chemed ...,NaN
